In [1]:
# ============================================
# VOTING ENSEMBLE CLASSIFICATION
# Built-in Breast Cancer Dataset
# ============================================

# 1. Import libraries
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import VotingClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ============================================
# 2. Load built-in dataset
# ============================================

data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset shape:", X.shape)
print("Classes:", data.target_names)


# ============================================
# 3. Train-Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================
# 4. Feature Scaling
# ============================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ============================================
# 5. Create individual models
# ============================================

lr = LogisticRegression(max_iter=1000)

dt = DecisionTreeClassifier(
    random_state=42
)

knn = KNeighborsClassifier(
    n_neighbors=5
)


# ============================================
# 6. Train individual models
# ============================================

lr.fit(X_train_scaled, y_train)
dt.fit(X_train, y_train)
knn.fit(X_train_scaled, y_train)


# ============================================
# 7. Individual predictions
# ============================================

lr_pred = lr.predict(X_test_scaled)
dt_pred = dt.predict(X_test)
knn_pred = knn.predict(X_test_scaled)


# ============================================
# 8. Individual model accuracy
# ============================================

print("\nIndividual Model Accuracy")

print("Logistic Regression:",
      accuracy_score(y_test, lr_pred))

print("Decision Tree:",
      accuracy_score(y_test, dt_pred))

print("KNN:",
      accuracy_score(y_test, knn_pred))


# ============================================
# 9. HARD VOTING
# ============================================

hard_voting = VotingClassifier(
    estimators=[
        ('lr', lr),
        ('dt', dt),
        ('knn', knn)
    ],
    voting='hard'
)

# Important:
# VotingClassifier will train the models again
# using the data we provide here.

hard_voting.fit(
    X_train_scaled,
    y_train
)

hard_pred = hard_voting.predict(X_test_scaled)


# ============================================
# 10. HARD VOTING RESULTS
# ============================================

print("\nHard Voting Accuracy:")

print(
    accuracy_score(
        y_test,
        hard_pred
    )
)


# ============================================
# 11. SOFT VOTING
# ============================================

soft_voting = VotingClassifier(
    estimators=[
        ('lr', lr),
        ('dt', dt),
        ('knn', knn)
    ],
    voting='soft'
)

soft_voting.fit(
    X_train_scaled,
    y_train
)

soft_pred = soft_voting.predict(X_test_scaled)


# ============================================
# 12. SOFT VOTING RESULTS
# ============================================

print("\nSoft Voting Accuracy:")

print(
    accuracy_score(
        y_test,
        soft_pred
    )
)


# ============================================
# 13. Classification Report
# ============================================

print("\nClassification Report - Soft Voting")

print(
    classification_report(
        y_test,
        soft_pred,
        target_names=data.target_names
    )
)


# ============================================
# 14. Confusion Matrix
# ============================================

print("\nConfusion Matrix")

print(
    confusion_matrix(
        y_test,
        soft_pred
    )
)

Dataset shape: (569, 30)
Classes: ['malignant' 'benign']

Individual Model Accuracy
Logistic Regression: 0.9824561403508771
Decision Tree: 0.9122807017543859
KNN: 0.956140350877193

Hard Voting Accuracy:
0.9824561403508771

Soft Voting Accuracy:
0.9649122807017544

Classification Report - Soft Voting
              precision    recall  f1-score   support

   malignant       0.97      0.93      0.95        42
      benign       0.96      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114


Confusion Matrix
[[39  3]
 [ 1 71]]


Pipeline


In [2]:
from sklearn.pipeline import Pipeline

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

dt_pipeline = Pipeline([
    ('model', DecisionTreeClassifier(random_state=42))
])


voting = VotingClassifier(
    estimators=[
        ('lr', lr_pipeline),
        ('knn', knn_pipeline),
        ('dt', dt_pipeline)
    ],
    voting='soft'
)

voting.fit(X_train, y_train)

y_pred = voting.predict(X_test)

print("Voting Accuracy:",
      accuracy_score(y_test, y_pred))

Voting Accuracy: 0.9649122807017544


In [4]:
voting = VotingClassifier(
    estimators=[
        ('lr', lr_pipeline),
        ('knn', knn_pipeline),
        ('dt', dt_pipeline)
    ],
    voting='hard'
)

voting.fit(X_train, y_train)

y_pred = voting.predict(X_test)

print("Voting Accuracy:",
      accuracy_score(y_test, y_pred))

Voting Accuracy: 0.9824561403508771
